# 06 Forward Test


In [7]:
pip install alpaca-py



   -------------------- ------------------- 1/2 [alpaca-py]
   -------------------- ------------------- 1/2 [alpaca-py]
   -------------------- ------------------- 1/2 [alpaca-py]
   ---------------------------------------- 2/2 [alpaca-py]



In [1]:
from pathlib import Path
import sys
import os
from datetime import datetime, timedelta, timezone

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: c:\Users\vidhi\Downloads\pro_crypto_trading_system\pro_crypto_trading_system


## 1. Load Alpaca paper API keys

In [5]:
from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / ".env")

ALPACA_API_KEY = os.getenv("ALPACA_API_KEY")
ALPACA_SECRET_KEY = os.getenv("ALPACA_SECRET_KEY")

if not ALPACA_API_KEY or not ALPACA_SECRET_KEY:
    raise ValueError(
        "Missing Alpaca API keys. Create a .env file with ALPACA_API_KEY and ALPACA_SECRET_KEY."
    )

print("Alpaca API key loaded:", ALPACA_API_KEY[:4] + "..." + ALPACA_API_KEY[-4:])
print("Secret key loaded:", True)

Alpaca API key loaded: PKPJ...PZJJ
Secret key loaded: True


## 2. Connect to Alpaca Paper Trading

In [8]:
from alpaca.trading.client import TradingClient
from alpaca.trading.requests import MarketOrderRequest
from alpaca.trading.enums import OrderSide, TimeInForce

trading_client = TradingClient(
    api_key=ALPACA_API_KEY,
    secret_key=ALPACA_SECRET_KEY,
    paper=True
)

account = trading_client.get_account()

print("Account status:", account.status)
print("Buying power:", account.buying_power)
print("Cash:", account.cash)
print("Portfolio value:", account.portfolio_value)

Account status: AccountStatus.ACTIVE
Buying power: 400000
Cash: 100000
Portfolio value: 100000


## 3. Configure crypto universe

Alpaca crypto symbols use formats like `BTC/USD`, while your backtest used Binance-style labels like `BTCUSDT`.

In [9]:
SYMBOL_MAP = {
    "BTCUSDT": "BTC/USD",
    "ETHUSDT": "ETH/USD",
    "SOLUSDT": "SOL/USD",
    "DOGEUSDT": "DOGE/USD",
    "LTCUSDT": "LTC/USD",
}

ALPACA_SYMBOLS = list(SYMBOL_MAP.values())

print(SYMBOL_MAP)

{'BTCUSDT': 'BTC/USD', 'ETHUSDT': 'ETH/USD', 'SOLUSDT': 'SOL/USD', 'DOGEUSDT': 'DOGE/USD', 'LTCUSDT': 'LTC/USD'}


## 4. Pull recent 5-minute crypto bars from Alpaca

In [10]:
from alpaca.data.historical import CryptoHistoricalDataClient
from alpaca.data.requests import CryptoBarsRequest
from alpaca.data.timeframe import TimeFrame, TimeFrameUnit

data_client = CryptoHistoricalDataClient()

end = datetime.now(timezone.utc)
start = end - timedelta(days=10)

request_params = CryptoBarsRequest(
    symbol_or_symbols=ALPACA_SYMBOLS,
    timeframe=TimeFrame(5, TimeFrameUnit.Minute),
    start=start,
    end=end
)

bars = data_client.get_crypto_bars(request_params).df

print(type(bars))
print(bars.head())
print(bars.tail())

<class 'pandas.core.frame.DataFrame'>
                                        open       high         low  \
symbol  timestamp                                                     
BTC/USD 2026-06-22 16:45:00+00:00  64502.680  64595.720  64431.8890   
        2026-06-22 16:50:00+00:00  64525.500  64712.560  64516.7240   
        2026-06-22 16:55:00+00:00  64693.535  64732.250  64634.6840   
        2026-06-22 17:00:00+00:00  64629.600  64699.105  64571.9385   
        2026-06-22 17:05:00+00:00  64651.880  64669.172  64522.9200   

                                        close    volume  trade_count  \
symbol  timestamp                                                      
BTC/USD 2026-06-22 16:45:00+00:00  64510.6800  0.000075          1.0   
        2026-06-22 16:50:00+00:00  64676.2775  0.005790          3.0   
        2026-06-22 16:55:00+00:00  64640.7270  0.000000          0.0   
        2026-06-22 17:00:00+00:00  64663.9695  0.025159          6.0   
        2026-06-22 17:05:00+00:0

## 5. Convert Alpaca bars

In [11]:
def alpaca_bars_to_project_format(bars_df: pd.DataFrame) -> pd.DataFrame:
    df = bars_df.reset_index().copy()

    df = df.rename(columns={
        "timestamp": "Datetime",
        "open": "Open",
        "high": "High",
        "low": "Low",
        "close": "Close",
        "volume": "Volume",
    })

    reverse_map = {v: k for k, v in SYMBOL_MAP.items()}
    df["symbol"] = df["symbol"].map(reverse_map)

    keep_cols = ["Datetime", "symbol", "Open", "High", "Low", "Close", "Volume"]
    df = df[keep_cols].dropna()
    df["Datetime"] = pd.to_datetime(df["Datetime"], utc=True)
    df = df.sort_values(["symbol", "Datetime"]).reset_index(drop=True)
    return df

recent_raw = alpaca_bars_to_project_format(bars)

print(recent_raw.shape)
print(recent_raw.head())
print(recent_raw.groupby("symbol").size())

(14242, 7)
                   Datetime   symbol       Open       High         Low  \
0 2026-06-22 16:45:00+00:00  BTCUSDT  64502.680  64595.720  64431.8890   
1 2026-06-22 16:50:00+00:00  BTCUSDT  64525.500  64712.560  64516.7240   
2 2026-06-22 16:55:00+00:00  BTCUSDT  64693.535  64732.250  64634.6840   
3 2026-06-22 17:00:00+00:00  BTCUSDT  64629.600  64699.105  64571.9385   
4 2026-06-22 17:05:00+00:00  BTCUSDT  64651.880  64669.172  64522.9200   

        Close    Volume  
0  64510.6800  0.000075  
1  64676.2775  0.005790  
2  64640.7270  0.000000  
3  64663.9695  0.025159  
4  64563.9600  0.003682  
symbol
BTCUSDT     2880
DOGEUSDT    2832
ETHUSDT     2837
LTCUSDT     2823
SOLUSDT     2870
dtype: int64


## 6. Engineer forward-test features

In [12]:
def engineer_forward_features(raw: pd.DataFrame) -> pd.DataFrame:
    df = raw.copy().sort_values(["symbol", "Datetime"]).reset_index(drop=True)
    g = df.groupby("symbol", group_keys=False)

    df["ret_5m"] = g["Close"].pct_change()
    df["log_ret_5m"] = g["Close"].transform(lambda s: np.log(s).diff())

    df["momentum_12"] = g["Close"].pct_change(12)
    df["momentum_48"] = g["Close"].pct_change(48)
    df["momentum_288"] = g["Close"].pct_change(288)

    df["ma_24"] = g["Close"].transform(lambda s: s.rolling(24).mean())
    df["ma_96"] = g["Close"].transform(lambda s: s.rolling(96).mean())
    df["ma_spread"] = df["ma_24"] / df["ma_96"] - 1

    df["vol_48"] = g["log_ret_5m"].transform(lambda s: s.rolling(48).std())
    df["vol_288"] = g["log_ret_5m"].transform(lambda s: s.rolling(288).std())

    df["cs_momentum_rank"] = df.groupby("Datetime")["momentum_48"].rank(pct=True)
    df["cs_vol_rank"] = df.groupby("Datetime")["vol_288"].rank(pct=True)

    df["raw_signal"] = (
        0.50 * df["cs_momentum_rank"]
        + 0.30 * df["ma_spread"].rank(pct=True)
        - 0.20 * df["cs_vol_rank"]
    )

    df["signal_rank"] = df.groupby("Datetime")["raw_signal"].rank(pct=True)

    return df.dropna().reset_index(drop=True)

forward_features = engineer_forward_features(recent_raw)

print(forward_features.shape)
forward_features.tail()

(12802, 21)


,Datetime,symbol,Open,High,Low,Close,Volume,ret_5m,log_ret_5m,momentum_12,...,momentum_288,ma_24,ma_96,ma_spread,vol_48,vol_288,cs_momentum_rank,cs_vol_rank,raw_signal,signal_rank
12797,2026-07-02 16:20:00+00:00,SOLUSDT,80.8500,80.8500,80.4800,80.70055,0.775402,0.000317,0.000317,0.000999,...,0.044210,80.560031,80.635054,-0.000930,0.002784,0.002225,0.2,1.0,0.032098,0.2
12798,2026-07-02 16:25:00+00:00,SOLUSDT,80.8611,80.8611,80.2168,80.42400,0.742561,-0.003427,-0.003433,-0.002326,...,0.034903,80.556135,80.659255,-0.001278,0.002769,0.002213,0.2,1.0,0.025539,0.2
12799,2026-07-02 16:30:00+00:00,SOLUSDT,80.4772,80.6807,80.2784,80.27840,0.745113,-0.001810,-0.001812,-0.004021,...,0.035293,80.548006,80.680104,-0.001637,0.002586,0.002211,0.2,1.0,0.018501,0.2
12800,2026-07-02 16:35:00+00:00,SOLUSDT,80.2315,80.4420,80.0810,80.19880,5.674262,-0.000992,-0.000992,-0.004601,...,0.035057,80.532538,80.699945,-0.002074,0.002584,0.002212,0.2,1.0,0.009566,0.2
12801,2026-07-02 16:40:00+00:00,SOLUSDT,80.2560,80.2846,80.2560,80.28460,0.000000,0.001070,0.001069,-0.004998,...,0.034633,80.526042,80.720780,-0.002412,0.002587,0.002211,0.2,1.0,0.003922,0.2


## 7. Generate latest signals

In [13]:
LONG_THRESHOLD = 0.80
EXIT_THRESHOLD = 0.55
MAX_SYMBOLS = 2

latest_time = forward_features["Datetime"].max()
latest_slice = forward_features[forward_features["Datetime"] == latest_time].copy()

latest_slice["signal"] = "HOLD"

long_candidates = (
    latest_slice[latest_slice["signal_rank"] >= LONG_THRESHOLD]
    .sort_values("signal_rank", ascending=False)
    .head(MAX_SYMBOLS)
)

latest_slice.loc[latest_slice["symbol"].isin(long_candidates["symbol"]), "signal"] = "BUY"
latest_slice.loc[latest_slice["signal_rank"] < EXIT_THRESHOLD, "signal"] = "SELL"

latest_signals = latest_slice[[
    "Datetime", "symbol", "Close", "signal", "signal_rank",
    "momentum_48", "ma_spread", "vol_288"
]].sort_values("signal_rank", ascending=False)

latest_signals

,Datetime,symbol,Close,signal,signal_rank,momentum_48,ma_spread,vol_288
7684,2026-07-02 16:40:00+00:00,ETHUSDT,1693.771500,BUY,1.0,0.021347,0.016627,0.002011
10219,2026-07-02 16:40:00+00:00,LTCUSDT,43.330600,BUY,0.8,0.007255,0.002814,0.001736
5135,2026-07-02 16:40:00+00:00,DOGEUSDT,0.074277,HOLD,0.6,0.000537,0.003457,0.001741
2591,2026-07-02 16:40:00+00:00,BTCUSDT,61339.523500,SELL,0.4,0.000349,0.003161,0.001441
12801,2026-07-02 16:40:00+00:00,SOLUSDT,80.284600,SELL,0.2,-0.010712,-0.002412,0.002211


## 8. Read current Alpaca positions

In [14]:
def get_current_positions():
    positions = trading_client.get_all_positions()
    rows = []

    for p in positions:
        rows.append({
            "alpaca_symbol": p.symbol,
            "qty": float(p.qty),
            "market_value": float(p.market_value),
            "current_price": float(p.current_price),
            "unrealized_pl": float(p.unrealized_pl),
        })

    return pd.DataFrame(rows)

positions_df = get_current_positions()
positions_df

""


## 9. Convert signals into paper orders


In [18]:
DRY_RUN = False
ORDER_NOTIONAL_USD = 100.0

current_positions = {}
if not positions_df.empty:
    for _, row in positions_df.iterrows():
        current_positions[row["alpaca_symbol"].replace("/", "")] = row["qty"]

print("Current positions:", current_positions)

Current positions: {}


In [19]:
def submit_market_order(alpaca_symbol: str, side: str, notional: float = None, qty: float = None):
    if side.upper() == "BUY":
        order_side = OrderSide.BUY
    elif side.upper() == "SELL":
        order_side = OrderSide.SELL
    else:
        raise ValueError("side must be BUY or SELL")

    if notional is not None:
        order_data = MarketOrderRequest(
            symbol=alpaca_symbol,
            notional=notional,
            side=order_side,
            time_in_force=TimeInForce.GTC
        )
    elif qty is not None:
        order_data = MarketOrderRequest(
            symbol=alpaca_symbol,
            qty=qty,
            side=order_side,
            time_in_force=TimeInForce.GTC
        )
    else:
        raise ValueError("Either notional or qty must be supplied.")

    if DRY_RUN:
        print("[DRY RUN] Would submit:", order_data)
        return None

    return trading_client.submit_order(order_data)

submitted_orders = []

for _, row in latest_signals.iterrows():
    project_symbol = row["symbol"]
    alpaca_symbol = SYMBOL_MAP[project_symbol]
    alpaca_position_key = alpaca_symbol.replace("/", "")
    signal = row["signal"]

    if signal == "BUY":
        already_held = current_positions.get(alpaca_position_key, 0.0) > 0
        if not already_held:
            order = submit_market_order(
                alpaca_symbol=alpaca_symbol,
                side="BUY",
                notional=ORDER_NOTIONAL_USD
            )
            submitted_orders.append({
                "Datetime": row["Datetime"],
                "symbol": project_symbol,
                "alpaca_symbol": alpaca_symbol,
                "signal": signal,
                "notional": ORDER_NOTIONAL_USD,
                "order_object": str(order)
            })

    elif signal == "SELL":
        held_qty = current_positions.get(alpaca_position_key, 0.0)
        if held_qty > 0:
            order = submit_market_order(
                alpaca_symbol=alpaca_symbol,
                side="SELL",
                qty=held_qty
            )
            submitted_orders.append({
                "Datetime": row["Datetime"],
                "symbol": project_symbol,
                "alpaca_symbol": alpaca_symbol,
                "signal": signal,
                "qty": held_qty,
                "order_object": str(order)
            })

submitted_orders_df = pd.DataFrame(submitted_orders)
submitted_orders_df

,Datetime,symbol,alpaca_symbol,signal,notional,order_object
0,2026-07-02 16:40:00+00:00,ETHUSDT,ETH/USD,BUY,100.0,id=UUID('7b5355b9-2425-44b6-8533-a10880a6c3d2'...
1,2026-07-02 16:40:00+00:00,LTCUSDT,LTC/USD,BUY,100.0,id=UUID('9504bc5c-44f3-4baf-93d7-f46820bd2393'...


## 10. Save forward-test logs

In [20]:
LOG_DIR = PROJECT_ROOT / "outputs" / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

signals_path = LOG_DIR / f"alpaca_forward_signals_{timestamp}.csv"
orders_path = LOG_DIR / f"alpaca_forward_orders_{timestamp}.csv"

latest_signals.to_csv(signals_path, index=False)
submitted_orders_df.to_csv(orders_path, index=False)

print("Saved signals:", signals_path)
print("Saved orders:", orders_path)

Saved signals: c:\Users\vidhi\Downloads\pro_crypto_trading_system\pro_crypto_trading_system\outputs\logs\alpaca_forward_signals_20260702_164519.csv
Saved orders: c:\Users\vidhi\Downloads\pro_crypto_trading_system\pro_crypto_trading_system\outputs\logs\alpaca_forward_orders_20260702_164519.csv
